# Baseline 2 — MFCC-focused feature space

Baseline 1 ([02_baseline_recommender_1_tim.ipynb](02_baseline_recommender_1_tim.ipynb)) used all 518
standardized FMA audio features. Its per-family contribution analysis showed that a handful of
families (mfcc, chroma) dominate the cosine score mainly because they have the most columns, not
necessarily because they carry the most useful similarity signal.

A colleague's hypothesis: **mfcc together with rhythm** could be a stronger, more targeted feature
set than "everything". This notebook is the controlled test of the mfcc half of that idea.

**Scope note:** FMA's precomputed `features.csv` has no tempo/beat/rhythm family at all (only
mfcc, chroma_cens/cqt/stft, spectral_contrast, tonnetz, rmse, spectral_bandwidth/centroid/rolloff,
zcr). Testing "mfcc + rhythm" properly would require extracting tempo/beat features from the raw
audio with librosa, which is out of scope for this pass. Here we test the achievable half of the
hypothesis now: **does restricting to mfcc (optionally + tonnetz, the strongest per-feature
contributor in Baseline 1) match or beat the full 518-feature space?**

Same pipeline, same excluded-artist logic, same 10 fixed seed tracks and rating rubric as
Baseline 1, so results are directly comparable.

`03_baseline_recommender_pca146_tim.ipynb` is an untitled duplicate of Baseline 1 with no PCA step
— treat it as scratch, not as an established PCA result.

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
tracks = pd.read_csv(
    "../data/raw/fma_metadata/tracks.csv",
    index_col=0,
    header=[0, 1]
)

features = pd.read_csv(
    "../data/raw/fma_metadata/features.csv",
    index_col=0,
    header=[0, 1, 2]
)

In [3]:
small_tracks = tracks[
    tracks[("set", "subset")] == "small"
].copy()

small_features = features.loc[small_tracks.index].copy()

small_tracks.shape, small_features.shape

((8000, 52), (8000, 518))

## Build feature-family variants

Three candidate feature spaces, each standardized independently:

- `full_518` — Baseline 1's full feature space (reference)
- `mfcc_only` — 140 mfcc columns only
- `mfcc_tonnetz` — mfcc + tonnetz (182 columns); tonnetz had the highest mean
  per-feature contribution in Baseline 1's family analysis

In [4]:
# Build each variant as a plain ndarray before scaling. mfcc and tonnetz share
# (statistic, coefficient) labels (e.g. ("mean", "01")), so concatenating them
# as DataFrames produces duplicate column names that trip up sklearn's
# feature-name validation — go via .to_numpy() and hstack instead.

feature_variants_raw = {
    "full_518": small_features.to_numpy(),
    "mfcc_only": small_features["mfcc"].to_numpy(),
    "mfcc_tonnetz": np.hstack(
        [
            small_features["mfcc"].to_numpy(),
            small_features["tonnetz"].to_numpy(),
        ]
    ),
}

X_variants = {
    name: StandardScaler().fit_transform(X_raw)
    for name, X_raw in feature_variants_raw.items()
}

{name: X.shape for name, X in X_variants.items()}

{'full_518': (8000, 518),
 'mfcc_only': (8000, 140),
 'mfcc_tonnetz': (8000, 182)}

## Recommendation and evaluation functions

Same logic as Baseline 1, parameterised on the feature matrix so it can be reused across variants.

In [5]:
def recommend_similar_tracks(X, track_id, n=10):

    if track_id not in small_tracks.index:
        raise ValueError("Track ID not found in FMA Small.")

    position = small_tracks.index.get_loc(track_id)

    seed_artist = small_tracks.loc[
        track_id, ("artist", "name")
    ]

    model = NearestNeighbors(metric="cosine", algorithm="brute")
    model.fit(X)

    # Retrieve more candidates than needed because
    # some will be removed by the artist filter
    candidate_count = min(n + 50, len(small_tracks))

    distances, indices = model.kneighbors(
        X[position].reshape(1, -1),
        n_neighbors=candidate_count
    )

    rows = []

    for idx, distance in zip(indices[0], distances[0]):

        candidate_id = small_tracks.index[idx]

        if candidate_id == track_id:
            continue

        candidate_artist = small_tracks.loc[
            candidate_id, ("artist", "name")
        ]

        if candidate_artist == seed_artist:
            continue

        rows.append({
            "track_id": candidate_id,
            "artist": candidate_artist,
            "title": small_tracks.loc[candidate_id, ("track", "title")],
            "genre": small_tracks.loc[candidate_id, ("track", "genre_top")],
            "cosine_distance": distance,
            "similarity": 1 - distance
        })

        if len(rows) == n:
            break

    return pd.DataFrame(rows).set_index("track_id")

In [6]:
def genre_agreement_at_k_exclude_artist(X, tracks_df, k=10, candidate_pool=100):

    model = NearestNeighbors(metric="cosine", algorithm="brute")
    model.fit(X)

    n_candidates = min(candidate_pool + 1, len(tracks_df))
    _, indices = model.kneighbors(X, n_neighbors=n_candidates)

    genres = tracks_df[("track", "genre_top")].to_numpy()
    artists = tracks_df[("artist", "name")].to_numpy()

    scores = []

    for i in range(len(tracks_df)):

        if pd.isna(genres[i]):
            continue

        valid_neighbours = []

        for neighbour_idx in indices[i]:
            if neighbour_idx == i:
                continue
            if artists[neighbour_idx] == artists[i]:
                continue
            valid_neighbours.append(neighbour_idx)
            if len(valid_neighbours) == k:
                break

        if not valid_neighbours:
            continue

        neighbour_genres = genres[valid_neighbours]
        valid_genres = ~pd.isna(neighbour_genres)

        if valid_genres.sum() == 0:
            continue

        scores.append((neighbour_genres[valid_genres] == genres[i]).mean())

    return float(np.mean(scores))

## Quantitative comparison: genre agreement@10

Fast, automatable proxy — computed across all 8,000 tracks in FMA Small for every variant.

In [7]:
genre_agreement_results = {}

for name, X in X_variants.items():
    t0 = time.time()
    score = genre_agreement_at_k_exclude_artist(X, small_tracks, k=10)
    elapsed = time.time() - t0
    genre_agreement_results[name] = {
        "n_features": X.shape[1],
        "genre_agreement_at_10": score,
        "seconds": elapsed,
    }

genre_agreement_df = pd.DataFrame(genre_agreement_results).T
genre_agreement_df

,n_features,genre_agreement_at_10,seconds
full_518,518.0,0.343425,0.969081
mfcc_only,140.0,0.343712,0.670457
mfcc_tonnetz,182.0,0.343125,0.716079


**Result: genre agreement@10 is essentially flat across all three variants** (full-518, mfcc-only,
mfcc+tonnetz all land within ~0.001 of each other, around 0.343). Dropping ~340 of the 518 features
neither hurts nor helps this metric.

That's a weaker result than hoped, but it's consistent with Baseline 1's other finding: cosine
similarity (on the full feature set) had almost no correlation with human similarity ratings
(Pearson r = 0.085). Genre agreement is a coarse, automatable proxy — it may simply not be
sensitive to the kind of similarity difference a smaller, more targeted feature set would produce.
The real test is the qualitative comparison below, using the same seeds Baseline 1 was rated on.

## Qualitative comparison setup

Same 10 fixed seed tracks and 1–5 rubric as Baseline 1 (see [02_baseline_recommender_1_tim.ipynb](02_baseline_recommender_1_tim.ipynb) for the full rubric and connection-type definitions). Recommendations here are generated from the `mfcc_only` variant, per the scoped-down comparison for this pass.

In [8]:
test_track_ids = [
    140,
    54033,
    59876,
    42234,
    68893,
    43883,
    123978,
    149701,
    63471,
    17608
]

[id_ for id_ in test_track_ids if id_ not in small_tracks.index]

[]

In [9]:
evaluation_rows = []

for seed_id in test_track_ids:

    seed = small_tracks.loc[seed_id]

    recommendations = recommend_similar_tracks(
        X_variants["mfcc_only"],
        seed_id,
        n=5
    )

    for rec_id, rec in recommendations.iterrows():

        evaluation_rows.append({
            "seed_track_id": seed_id,
            "seed_artist": seed[("artist", "name")],
            "seed_title": seed[("track", "title")],
            "seed_genre": seed[("track", "genre_top")],

            "recommended_track_id": rec_id,
            "recommended_artist": rec["artist"],
            "recommended_title": rec["title"],
            "recommended_genre": rec["genre"],

            "cosine_similarity": rec["similarity"],

            "manual_similarity_1_5": np.nan,
            "connection_type": "",
            "notes": ""
        })

manual_eval_mfcc = pd.DataFrame(evaluation_rows)
manual_eval_mfcc.shape

(50, 12)

In [10]:
manual_eval_mfcc.to_csv(
    "../data/processed/baseline2_mfcc_manual_evaluation.csv",
    index=False
)

manual_eval_mfcc.head(10)

,seed_track_id,seed_artist,seed_title,seed_genre,recommended_track_id,recommended_artist,recommended_title,recommended_genre,cosine_similarity,manual_similarity_1_5,connection_type,notes
0,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,11268,Le Tigre,Fake French,Rock,0.563668,NaN,,
1,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,11299,Chuck D,No Meaning No (feat. Fine Arts Militia),Hip-Hop,0.536556,NaN,,
2,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,62186,Daouda,La femme de mon patron,International,0.535449,NaN,,
3,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,27804,Scomber,|Two Sides| Why? (ft. Colin Mutchler & Myklean...,Folk,0.535197,NaN,,
4,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,49857,My Bubba & Mi,After you,Folk,0.508813,NaN,,
5,54033,alright lover,Snow Wave,Pop,122510,Squire Tuck,Solitude Amongst the Wild Meadow Flowers,Instrumental,0.733767,NaN,,
6,54033,alright lover,Snow Wave,Pop,68540,Benboncan,Murmur,Experimental,0.732780,NaN,,
7,54033,alright lover,Snow Wave,Pop,133439,TRG Banks,Horse ride 4,Instrumental,0.730454,NaN,,
8,54033,alright lover,Snow Wave,Pop,125238,Derek Baron,Palmillas,Experimental,0.718010,NaN,,
9,54033,alright lover,Snow Wave,Pop,133437,TRG Banks,Horse ride 3,Instrumental,0.717681,NaN,,


## Listen and rate

Use the exact same rubric as Baseline 1 (1–5 similarity, `connection_type` tags: rhythm, timbre,
mood, instrumentation, energy, production, harmony, vocals, cross-genre, unclear). Rate blind to
the cosine score where possible.

**Do not re-run the cells above once rating starts** — that regenerates `manual_eval_mfcc` from
scratch and overwrites the csv, wiping any ratings entered in the meantime (this is what happened
to Baseline 1's data the first time round). Rate directly in
`../data/processed/baseline2_mfcc_manual_evaluation.csv` and only re-run the import cell below.

Once filled in, re-import it below and compute the same summary stats as Baseline 1 (mean rating,
% ≥ 3, % ≥ 4, correlation with cosine similarity) for a direct comparison against Baseline 1's
2.74 / 5 mean.

In [11]:
from IPython.display import Audio, display
from pathlib import Path


def get_fma_audio_path(track_id):
    track_id_str = f"{track_id:06d}"
    return (
        Path("../data/raw/fma_small")
        / track_id_str[:3]
        / f"{track_id_str}.mp3"
    )


def listen_to_pair(seed_id, rec_id):
    print("SEED")
    print(small_tracks.loc[seed_id, [("artist", "name"), ("track", "title")]])
    display(Audio(get_fma_audio_path(seed_id)))

    print("\nRECOMMENDATION")
    print(small_tracks.loc[rec_id, [("artist", "name"), ("track", "title")]])
    display(Audio(get_fma_audio_path(rec_id)))

## Leave the audio section above!

Add one `listen_to_pair(seed_id, rec_id)` cell per row in `baseline2_mfcc_manual_evaluation.csv`, listen, then fill in `manual_similarity_1_5` and `connection_type` directly in the csv (same convention as Baseline 1).

In [12]:
# listen_to_pair(seed_id, rec_id)

## Import ratings back in and compute Baseline 2 summary

Run once `manual_similarity_1_5` has been filled in for all 50 rows.

In [13]:
manual_eval_mfcc = pd.read_csv(
    "../data/processed/baseline2_mfcc_manual_evaluation.csv",
    sep=None,
    engine="python"
)

# Spreadsheet apps used to enter ratings can re-save with a different
# delimiter and duplicate the "connection_type" header (see Baseline 1) —
# recover the "notes" column if pandas had to de-duplicate it on read.
if "connection_type.1" in manual_eval_mfcc.columns:
    manual_eval_mfcc = manual_eval_mfcc.rename(
        columns={"connection_type.1": "notes"}
    )

manual_eval_mfcc["manual_similarity_1_5"].isna().sum()

np.int64(50)

In [14]:
mean_manual_similarity = manual_eval_mfcc["manual_similarity_1_5"].mean()

pct_3_or_more = (manual_eval_mfcc["manual_similarity_1_5"] >= 3).mean()
pct_4_or_more = (manual_eval_mfcc["manual_similarity_1_5"] >= 4).mean()

print("Mean manual similarity:", mean_manual_similarity)
print("% rated 3 or more:", pct_3_or_more)
print("% rated 4 or more:", pct_4_or_more)

manual_eval_mfcc[["cosine_similarity", "manual_similarity_1_5"]].corr()

Mean manual similarity: nan
% rated 3 or more: 0.0
% rated 4 or more: 0.0


,cosine_similarity,manual_similarity_1_5
cosine_similarity,1.0,NaN
manual_similarity_1_5,NaN,NaN


## Baseline 2 findings (fill in after rating)

### Quantitative evaluation
- Genre agreement@10 — full-518: **0.3434**, mfcc-only: **0.3437**, mfcc+tonnetz: **0.3431**
  (no meaningful difference; see note above on why this metric may not be sensitive enough)

### Manual evaluation (mfcc-only, same 10 seeds / 50 recs as Baseline 1)
- Mean human similarity: **TBD** (Baseline 1: 2.74 / 5)
- % rated ≥ 3: **TBD** (Baseline 1: 66%)
- % rated ≥ 4: **TBD** (Baseline 1: 16%)
- Pearson correlation, cosine vs. human rating: **TBD** (Baseline 1: 0.085)

### Interpretation
TBD once ratings are in. If mfcc-only matches or beats Baseline 1 on the human ratings despite
identical genre-agreement scores, that's evidence the automatable metric is too coarse to guide
feature selection and the manual rubric should stay the primary yardstick going forward. If it's
noticeably worse, that argues against trimming to mfcc alone without the rhythm half of the
original hypothesis.